# Modeling a Spiking Neuron: Parameter Tuning via Numerical Optimization

This notebook is your workspace for the whole task. Read `neuro_task.md` first if you haven't
already. It has the full writeup, the objective, and the resource list. This notebook is where
you'll actually run the code and answer the questions.

The model we're working with is a compact spiking neuron model:

```
dv/dt = 0.04v² + 5v + 140 - u - w + I(t)
du/dt = a(bv - u)
dw/dt = -kw

if v >= 30 mV:
    v <- c
    u <- u + d
    w <- w + e
```

`v` is the membrane voltage. `u` and `w` are internal state variables. `a, b, c, d, e` are the five parameters you'll be optimizing.
`k` is fixed at `0.05`. `I(t)` is the input current, provided as a time-series array (see Part 1).

---
## Part 1: Euler's Method

The cell below is a complete, working numerical simulation of the model above, using the
[Euler method](https://en.wikipedia.org/wiki/Euler_method).

You don't have to implement any code in this part. It's only for you to visualize how we can approximate a voltage-time graph from the differential equations and how changing parameters affects said graph. However, you are expected to understand what the code is doing and how Euler's method works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


target = pd.read_csv("spike_data.csv")
t_target = target["time_ms"].values
v_target = target["voltage_mV"].values

k = 0.05

# arbitrary parameters
a = 0.02
b = 0.25
c = -60.0
d = 6.0
e = 0.3

dt = 0.5      # time step, in ms
T  = 200.0    # total simulation time, in ms
steps = int(T / dt)

t_arr = np.arange(steps) * dt          # time value at each step
I_arr = np.where((t_arr >= 10.0) & (t_arr <= 100.0), 10.0, 0.0)

v = -70.0           
u = b * v          
w = 0.0              

t_history = []
v_history = []

for i in range(steps):
    t = t_arr[i]
    I = I_arr[i]    

    dv = (0.04 * v**2 + 5*v + 140 - u - w + I) * dt
    du = (a * (b*v - u)) * dt
    dw = (-k * w) * dt

    v = v + dv
    u = u + du
    w = w + dw

    v_recorded = v

    if v >= 30.0:
        v_recorded = 30.0 
        v = c
        u = u + d
        w = w + e

    t_history.append(t)
    v_history.append(v_recorded)

fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)

axes[0].plot(t_history, v_history, color="crimson")
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Voltage (mV)")
axes[0].set_title("Simulated neuron — default parameters")

axes[1].plot(t_target, v_target, color="black")
axes[1].set_xlabel("Time (ms)")
axes[1].set_title("Target trace (spike_data.csv)")

plt.tight_layout()
plt.show()

---
## Part 2A: Manual Parameter Tuning

`spike_data.csv` contains a target voltage trace from a neuron whose underlying `a, b, c, d, e`
you don't know. Your job in this section is to try and match it by hand, adjusting parameters and
watching what happens.

The same step-current `I_arr` used in Part 1 was used to generate the target so `k` is already
fixed and `I(t)` is already known. You only need to explore `a, b, c, d, e`.

Adjust the values below and re-run the cell. The target trace is overlaid in black so you can
see how close you're getting.

In [ ]:
def simulate(a, b, c, d, e, k=0.05, dt=0.5, T=200.0, v0=-70.0):

    steps = int(T / dt)
    t_arr_s = np.arange(steps) * dt
    I_arr_s = np.where((t_arr_s >= 10.0) & (t_arr_s <= 100.0), 10.0, 0.0)

    v = v0
    u = b * v0
    w = 0.0

    t_history = []
    v_history = []

    for i in range(steps):
        t = t_arr_s[i]
        I = I_arr_s[i]

        dv = (0.04 * v**2 + 5*v + 140 - u - w + I) * dt
        du = (a * (b*v - u)) * dt
        dw = (-k * w) * dt

        v = v + dv
        u = u + du
        w = w + dw

        v_recorded = v
        if v >= 30.0:
            v_recorded = 30.0
            v = c
            u = u + d
            w = w + e

        t_history.append(t)
        v_history.append(v_recorded)

    return np.array(t_history), np.array(v_history)


# you can play around with the values of these parameters
a_guess = 0.02
b_guess = 0.25
c_guess = -60.0
d_guess = 6.0
e_guess = 0.3

t_sim, v_sim = simulate(a_guess, b_guess, c_guess, d_guess, e_guess)

plt.figure(figsize=(10, 4))
plt.plot(t_target, v_target, color="black", label="target", alpha=0.6)
plt.plot(t_sim, v_sim, color="crimson", label="your guess")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.legend()
plt.title("Manual parameter matching")
plt.show()

**Questions: Part 2A**

1. After playing around with the parameters, what can you tell about each one? What do you think they represent?
2. Now try to actually match the black target trace, purely by trial and error. What values of the parameters
   lead to the most similar graph? 

---
## Part 2B: Algorithmic Parameter Tuning

Trial and error only gets you so far with five interacting parameters. Now you'll fit the model
properly using an optimization algorithm.

Before any optimization can happen, you have to figure out what you actually want to optimize. This target
is essentially what a loss function is. **The design of the loss function is entirely yours**. There 
are many reasonable approaches, and your reasoning for/understanding of the one you pick is arguably more important than 
how accurate your final graph is. Check the resources in `neuro_task.md` for a starting point, then explore beyond them.

Important Note: Please do not adjust your initial guesses in Part 2A to resemble parameter values obtained after using an
optimization algorithm. Parameter accuracy in Part 2A is pretty irrelevant. 

In [ ]:
from scipy.optimize import minimize

# function goes here

**Questions: Part 2B**

1. What loss function did you choose, and why?
2. Plot your optimized simulation against the target. How close is the match?

In [ ]:
# plot your optimized fit against the target here


---
## Part 3: Compare

You now have two sets of parameters: the ones you guessed by hand in Part 2A, and the ones the
optimizer found in Part 2B. This section is about comparing them.

In [ ]:
comparison = pd.DataFrame({
    "parameter": ["a", "b", "c", "d", "e"],
    "your_2A_guess": [a_guess, b_guess, c_guess, d_guess, e_guess],
    "optimized_2B": [None, None, None, None, None],  # fill in with optimized_params
})
comparison

**Questions: Part 3**

1. Which parameters were you closest on, and which were way off? Why do you think that is?
2. Looking at the optimized values, do your assumptions from Part 2A still hold up?
4. Flag anything else that stood out: an unexpected optimizer behavior, a parameter that
   didn't do what you thought, anything.